# ChaoSSM — Phase 0b: Map Redesign & Corridor Reachability

## What Phase 0 established

The smoke run was inconclusive on everything training-dependent, but it surfaced three
findings that do **not** depend on training length:

1. **§4.3 Option A cannot be chaotic.** For `φ(h) = sigmoid(W_mix·h + r·h(1−h))`, the Jacobian
   is `diag(σ'(z))·(W_mix + diag(r(1−2h)))`. Since `σ' ≤ 0.25` and `r ≤ 4`, their product
   cannot exceed 1.0, so `log σ_max ≤ ~0` **by construction**. A 24,000-sample sweep found a
   maximum of +0.018, attained only at the clamp boundary. Measured λ₁(φ) was −0.66 to −0.80
   with r sitting at 3.785 — well inside the chaotic range for the *scalar* map. The outer
   sigmoid absorbs the chaos.
2. **The training-time estimator is biased high by ~0.5 nats.** Single-step `σ_max` is an upper
   bound on the asymptotic λ₁ (they coincide in 1-D, not in N-D where directions rotate under
   the Jacobian product). Measured: λ_power ≈ −0.15 vs λ_Benettin(φ) ≈ −0.66.
3. **§4.4 regularizes the wrong operator.** λ(φ) ≈ −0.66 vs λ(full recurrence) ≈ −0.45 — a
   −0.25 nat gap created by the gate's `(1−g)⊙h` skip term, which the regularizer never sees.

Net effect: driving `λ_power(φ)` into [−0.05, +0.15] leaves the *actual* λ₁ of the full
recurrence near −0.3. **The corridor as specified targets ordered dynamics.**

## What 0b tests

One gating question: **can the corridor be reached at all, and by which map?**

- **Stage A — pre-flight reachability.** No training. For each map variant, sweep the attainable
  range of `log σ_max`. A variant whose ceiling is ≤ 0 is disqualified before a single gradient
  step. This costs seconds and is the check that should have run first.
- **Stage B — variant screen.** Train each surviving variant under the corridor regularizer.
- **Stage C — arm comparison.** On the winning variant only, run corridor / floss / free.

Three map variants and two regularizer targets are switchable:

| Variant | Form | σ_max ceiling |
|---|---|---|
| `original` | `sigmoid(W·h + r·h(1−h))` | `0.25 · 4 = 1.0` → λ ≤ 0 |
| `gain` | `sigmoid(γ·(W·h + r·h(1−h)))`, γ learnable | `(γ/4) · 4 = γ` → λ ≤ ln γ |
| `fold` | triangle-fold instead of squashing | `1 · 4 = 4` → λ ≤ ln 4 ≈ 1.386 |

`fold` also keeps the state in [0,1] by construction, which should fix the 70% clamp-activation
rate — clamped states contribute zero derivative and bias λ downward.

Estimators: `power` (single-step, the §6.3 default) and `window` (finite-time Lyapunov exponent
over W steps, which removes most of the single-step bias and is still differentiable).

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

torch 2.11.0+cu128 | device: cuda


In [2]:
def setup_results_dir(project="ChaoSSM", phase="phase0b"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True)
    print("results ->", base)
    return base

RESULTS = setup_results_dir()

def savefig(fig, name):
    p = os.path.join(RESULTS, name); fig.savefig(p, dpi=300, bbox_inches="tight"); print("saved", p)
def savetable(df, name):
    p = os.path.join(RESULTS, name); df.to_csv(p, index=False); print("saved", p)
def savejson(o, name):
    p = os.path.join(RESULTS, name)
    with open(p, "w") as f: json.dump(o, f, indent=2, default=str)
    print("saved", p)

Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase0b


## Configuration

Run with `SMOKE_TEST = True` once to confirm the pipeline, then set it to `False`.

Stage A is free — it runs regardless and takes seconds.

In [3]:
SMOKE_TEST = True    # <-- set False for the real run

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    copy_len: int = 16
    steps: int = 600
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    # regularizer
    lyap_alpha: float = 0.1        # raised from 0.01: Phase 0 showed 7.8e-05 share of loss
    lam_low: float = -0.05
    lam_high: float = 0.15
    reg_target: str = "full"       # "phi" (S4.4 as written) | "full" (recurrence)
    estimator: str = "window"      # "power" (S6.3) | "window" (FTLE)
    lyap_window: int = 8           # W for the FTLE estimator
    # instrumentation
    lyap_every: int = 10
    lyap_subsample: int = 6
    benettin_transient: int = 100
    benettin_measure: int = 300
    # experiment
    variants: tuple = ("original", "gain", "fold")
    gain_init: float = 4.0
    screen_seeds: int = 3          # Stage B
    seeds: int = 8                 # Stage C
    arms: tuple = ("corridor", "floss", "free")

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 60; CFG.screen_seeds = 1; CFG.seeds = 2
    CFG.benettin_measure = 60; CFG.benettin_transient = 20
    print("*** SMOKE TEST — pipeline check only, results not meaningful ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

*** SMOKE TEST — pipeline check only, results not meaningful ***
{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "copy_len": "16",
  "steps": "60",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "lyap_alpha": "0.1",
  "lam_low": "-0.05",
  "lam_high": "0.15",
  "reg_target": "full",
  "estimator": "window",
  "lyap_window": "8",
  "lyap_every": "10",
  "lyap_subsample": "6",
  "benettin_transient": "20",
  "benettin_measure": "60",
  "variants": "('original', 'gain', 'fold')",
  "gain_init": "4.0",
  "screen_seeds": "1",
  "seeds": "2",
  "arms": "('corridor', 'floss', 'free')"
}


## 1. Map variants

`fold` uses a triangle wave `x ↦ 1 − |((x mod 2) − 1)|`, which maps ℝ → [0,1] with derivative
magnitude exactly 1 everywhere. Unlike a sigmoid it bounds the state **without** contracting it,
so the logistic map's expansion survives into the composed map.

In [4]:
def triangle_fold(x):
    '''R -> [0,1], piecewise linear, |derivative| == 1 everywhere.'''
    y = torch.remainder(x, 2.0)
    return 1.0 - torch.abs(y - 1.0)


class ChaoticMap(nn.Module):
    def __init__(self, state_dim, variant="original", gain_init=4.0):
        super().__init__()
        self.variant = variant
        self.w_r = nn.Parameter(torch.zeros(state_dim))
        W = torch.empty(state_dim, state_dim); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        if variant == "gain":
            # gamma = softplus(w_gamma), initialized at gain_init
            inv = math.log(math.expm1(gain_init))
            self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
        self.register_buffer("clamp_hits", torch.zeros(()))
        self.register_buffer("clamp_total", torch.zeros(()))

    def get_r(self):
        return 3.57 + 0.43 * torch.sigmoid(self.w_r)

    def get_gamma(self):
        return F.softplus(self.w_gamma) if self.variant == "gain" else None

    def forward(self, h, count_clamp=False):
        if count_clamp:
            with torch.no_grad():
                o = ((h < 1e-6) | (h > 1 - 1e-6)).float()
                self.clamp_hits += o.sum(); self.clamp_total += o.numel()
        r = self.get_r()
        if self.variant == "fold":
            # no clamp: the fold keeps the state in [0,1] by construction
            u = h @ self.W_mix.T + r * h * (1 - h)
            return triangle_fold(u)
        h_c = torch.clamp(h, 1e-6, 1 - 1e-6)
        u = h_c @ self.W_mix.T + r * h_c * (1 - h_c)
        if self.variant == "gain":
            return torch.sigmoid(self.get_gamma() * u)
        return torch.sigmoid(u)


class ChaoSSMCell(nn.Module):
    def __init__(self, cfg: Config, variant: str):
        super().__init__()
        self.N, self.d, self.variant = cfg.state_dim, cfg.d_model, variant
        self.phi = ChaoticMap(cfg.state_dim, variant, cfg.gain_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))

    def step(self, h, x, count_clamp=False):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        h_new = g * self.phi(h @ self.A.T, count_clamp) + (1 - g) * h + x @ self.B.T
        if self.variant == "fold":
            h_new = triangle_fold(h_new)   # keep state in domain without squashing
        return h_new

    def init_state(self, batch, device):
        return torch.full((batch, self.N), 0.5, device=device)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg: Config, variant: str):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab + 2, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, variant)
        self.readout = nn.Linear(cfg.state_dim, cfg.vocab + 2)

    def forward(self, tokens, return_traj=False, count_clamp=False):
        B, T = tokens.shape
        x = self.emb(tokens)
        h = self.cell.init_state(B, tokens.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t], count_clamp)
            outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        logits = torch.stack(outs, 1)
        return (logits, torch.stack(traj, 1), x) if return_traj else logits

## 2. Stage A — pre-flight reachability (no training)

This is the check that gates everything. For each variant, sweep states and parameters and
record the attainable `log σ_max(J_φ)`. **A variant whose ceiling is ≤ 0 cannot host
edge-of-chaos dynamics at any hyperparameter setting**, and no amount of training will change
that.

Reported per variant: the maximum over the sweep, and the median at *typical* states (drawn
from the map's own output distribution, since that is where trajectories actually live). The
gap between the two matters — a ceiling reachable only in a measure-zero corner is not usable.

In [5]:
@torch.no_grad()
def _sigma_max_logs(variant, cfg, n_param=40, n_state=40, typical=False, seed=0):
    torch.manual_seed(seed)
    out = []
    for _ in range(n_param):
        m = ChaoticMap(cfg.state_dim, variant, cfg.gain_init).double()
        r = m.get_r()
        W = m.W_mix
        gam = m.get_gamma()
        for _ in range(n_state):
            if typical:
                # states where trajectories actually live: iterate the map a few times
                h = torch.rand(cfg.state_dim, dtype=torch.float64)
                for _ in range(20):
                    h = m(h)
            else:
                # aggressive sweep incl. domain edges
                pick = np.random.randint(4)
                h = [torch.rand(cfg.state_dim, dtype=torch.float64),
                     torch.full((cfg.state_dim,), 1e-6, dtype=torch.float64),
                     torch.full((cfg.state_dim,), 1 - 1e-6, dtype=torch.float64),
                     torch.rand(cfg.state_dim, dtype=torch.float64) * 1e-3][pick]
            hc = h if variant == "fold" else torch.clamp(h, 1e-6, 1 - 1e-6)
            u = hc @ W.T + r * hc * (1 - hc)
            inner = W + torch.diag(r * (1 - 2 * hc))
            if variant == "fold":
                d = torch.sign(1.0 - torch.remainder(u, 2.0))   # |slope| = 1
                J = torch.diag(d) @ inner
            elif variant == "gain":
                s = torch.sigmoid(gam * u); sp = gam * s * (1 - s)
                J = torch.diag(sp) @ inner
            else:
                s = torch.sigmoid(u); sp = s * (1 - s)
                J = torch.diag(sp) @ inner
            out.append(float(torch.log(torch.linalg.svdvals(J)[0] + 1e-12)))
    return np.array(out)


rows = []
for v in CFG.variants:
    agg = _sigma_max_logs(v, CFG, typical=False)
    typ = _sigma_max_logs(v, CFG, typical=True)
    rows.append(dict(variant=v,
                     ceiling_logsigma=float(agg.max()),
                     typical_median=float(np.median(typ)),
                     typical_p90=float(np.percentile(typ, 90)),
                     reachable=bool(agg.max() > 0),
                     usable=bool(np.percentile(typ, 90) > 0)))
PRE = pd.DataFrame(rows)
print(PRE.to_string(index=False))
savetable(PRE, "phase0b_stageA_reachability.csv")

fig, ax = plt.subplots(figsize=(7, 4))
xs = np.arange(len(PRE))
ax.bar(xs - 0.2, PRE.ceiling_logsigma, 0.4, label="ceiling (best case)")
ax.bar(xs + 0.2, PRE.typical_median, 0.4, label="typical state (median)")
ax.axhline(0, color="r", ls="--", label=r"$\lambda=0$ (edge of chaos)")
ax.set_xticks(xs); ax.set_xticklabels(PRE.variant)
ax.set(ylabel=r"$\log\sigma_{max}(J_\phi)$", title="Stage A — attainable expansion by map variant")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0b_stageA_reachability.png"); plt.close(fig)

VIABLE = PRE[PRE.usable].variant.tolist()
print("\nvariants that can reach lambda>0 at typical states:", VIABLE or "NONE")
if not VIABLE:
    print("!! No variant is viable. Do not proceed to training — redesign the map.")

 variant  ceiling_logsigma  typical_median  typical_p90  reachable  usable
original         -0.029173       -1.073203    -1.060001      False   False
    gain          1.357121        1.283614     1.293519       True    True
    fold          1.357121        1.311452     1.329278       True    True
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageA_reachability.csv
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageA_reachability.png

variants that can reach lambda>0 at typical states: ['gain', 'fold']


## 3. Estimators

`lambda_power` is §6.3's single-step estimate. `lambda_window` is the finite-time Lyapunov
exponent over W consecutive steps — `(1/W)·log σ_max(J_{t+W-1}···J_t)` — which captures the
directional rotation that the single-step estimate misses. Both are differentiable and usable
as training signals. `benettin_spectrum` remains the post-hoc reference.

In [6]:
from torch.func import jacrev, vmap

def _phi_jac_fn(model):
    A = model.cell.A
    return lambda z: model.cell.phi(z @ A.T)

def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]


def lambda_power(model, h_pts, x_pts, target):
    '''Single-step estimate (proposal S6.3).'''
    if target == "phi":
        J = vmap(jacrev(_phi_jac_fn(model)))(h_pts)
    else:
        J = torch.stack([jacrev(_full_jac_fn(model, x_pts[i]))(h_pts[i])
                         for i in range(h_pts.shape[0])])
    v = torch.randn(J.shape[0], J.shape[1], 1, device=J.device, dtype=J.dtype)
    for _ in range(5):
        v = torch.bmm(J.transpose(1, 2), torch.bmm(J, v))
        v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
    sm = torch.sqrt((torch.bmm(J, v) ** 2).sum(1)).squeeze(-1)
    return torch.log(sm + 1e-8).mean()


def lambda_window(model, traj, xseq, target, W, n_sample):
    '''Finite-time Lyapunov exponent over W consecutive steps.'''
    B, T, N = traj.shape
    if T <= W + 1:
        return lambda_power(model, traj[:, 0], xseq[:, 0], target)
    lams = []
    bi = torch.randperm(B)[:n_sample]
    for b in bi:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            h = traj[b, t0 + k]
            if target == "phi":
                J = jacrev(_phi_jac_fn(model))(h)
            else:
                J = jacrev(_full_jac_fn(model, xseq[b, t0 + k]))(h)
            M = J @ M
        sm = torch.linalg.matrix_norm(M, ord=2)
        lams.append(torch.log(sm + 1e-12) / W)
    return torch.stack(lams).mean()


@torch.no_grad()
def benettin_spectrum(step_fn, h0, inputs, N, T_tr, T_me):
    h = h0.clone()
    for t in range(T_tr): h = step_fn(h, inputs[t % len(inputs)])
    Q = torch.eye(N, device=h.device, dtype=h.dtype)
    lyap = torch.zeros(N, device=h.device, dtype=h.dtype)
    for t in range(T_me):
        x = inputs[(T_tr + t) % len(inputs)]
        J = jacrev(lambda hh: step_fn(hh, x))(h)
        Q = J @ Q; Q, R = torch.linalg.qr(Q)
        lyap += torch.log(torch.abs(torch.diagonal(R)) + 1e-12)
        h = step_fn(h, x)
    return (lyap / T_me).sort(descending=True).values


def kaplan_yorke(sp):
    s = sp.detach().cpu().numpy().astype(np.float64)
    c = np.cumsum(s); idx = np.where(c >= 0)[0]
    if len(idx) == 0: return 0.0
    j = int(idx[-1]) + 1
    if j >= len(s): return float(len(s))
    d = abs(s[j])
    return float(j) if d < 1e-12 else float(j + c[j - 1] / d)

### 3b. Estimator validation on a system with a known answer

In [7]:
def validate():
    r_val = 4.0; truth = math.log(2)
    f = lambda h: r_val * h * (1 - h)
    h = torch.tensor([0.31234], dtype=torch.float64); tr = []
    for _ in range(2000): h = f(h); tr.append(h.clone())
    pts = torch.stack(tr[500:]).reshape(-1, 1)
    J = vmap(jacrev(f))(pts)
    v = torch.randn(J.shape[0], 1, 1, dtype=torch.float64)
    for _ in range(5):
        v = torch.bmm(J.transpose(1, 2), torch.bmm(J, v)); v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
    lp = torch.log(torch.sqrt((torch.bmm(J, v) ** 2).sum(1)).squeeze(-1) + 1e-8).mean().item()
    sp = benettin_spectrum(lambda hh, x: f(hh), torch.tensor([0.31234], dtype=torch.float64),
                           [torch.zeros(1, dtype=torch.float64)], 1, 500, 2000)[0].item()
    df = pd.DataFrame([dict(truth=truth, power=lp, benettin=sp,
                            power_err=abs(lp - truth), benettin_err=abs(sp - truth))])
    return df

vdf = validate(); print(vdf.to_string(index=False))
ok = bool((vdf.power_err < 0.02).all() and (vdf.benettin_err < 0.02).all())
print(f"\nESTIMATOR VALIDATION: {'PASS' if ok else 'FAIL'}")
savetable(vdf, "phase0b_estimator_validation.csv")

   truth    power  benettin  power_err  benettin_err
0.693147 0.690642  0.692591   0.002506      0.000556

ESTIMATOR VALIDATION: PASS
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_estimator_validation.csv


## 4. Task and training

In [8]:
def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), V, device=device),
                     torch.full((batch,L), V+1, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt


def lyap_loss(lam, arm, cfg):
    if arm == "corridor":
        return cfg.lyap_alpha * (F.relu(lam - cfg.lam_high)**2 + F.relu(cfg.lam_low - lam)**2)
    if arm == "floss":  return cfg.lyap_alpha * lam**2
    if arm == "free":   return torch.zeros((), device=lam.device)
    raise ValueError(arm)


def train_one(variant, arm, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
    model = ChaoSSMLM(cfg, variant).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))
    hist, diverged = [], False

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        logits, traj, xseq = model(inp, return_traj=True, count_clamp=True)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)
        if cfg.estimator == "window":
            lam = lambda_window(model, traj, xseq, cfg.reg_target,
                                cfg.lyap_window, cfg.lyap_subsample)
        else:
            bi = torch.randperm(traj.shape[0], device=DEVICE)[:cfg.lyap_subsample]
            ti = torch.randint(0, traj.shape[1], (cfg.lyap_subsample,), device=DEVICE)
            lam = lambda_power(model, traj[bi, ti], xseq[bi, ti], cfg.reg_target)

        L_lyap = lyap_loss(lam, arm, cfg)
        opt.zero_grad(set_to_none=True); (L_task + L_lyap).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()
        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break
        if step % cfg.lyap_every == 0:
            with torch.no_grad():
                acc = (logits[:, cfg.copy_len+1:].argmax(-1) == tgt[:, cfg.copy_len+1:]).float().mean().item()
                cr = (model.cell.phi.clamp_hits / model.cell.phi.clamp_total.clamp(min=1)).item()
            hist.append(dict(step=step, variant=variant, arm=arm, seed=seed,
                             loss=L_task.item(), acc=acc, lam=lam.item(), gnorm=gn,
                             clamp_rate=cr, L_lyap=float(L_lyap.item()),
                             lyap_share=float(L_lyap.item()/max(abs(L_task.item()),1e-8))))

    # post-hoc reference spectra
    model.eval()
    with torch.no_grad():
        inp, _ = make_copy_batch(cfg, 1, DEVICE, gen)
        xs = [model.emb(inp[0, t]).double() for t in range(inp.shape[1])]
    cellf = model.cell.double(); h0 = cellf.init_state(1, DEVICE)[0].double()
    sp_full = benettin_spectrum(lambda hh, x: cellf.step(hh.unsqueeze(0), x.unsqueeze(0))[0],
                                h0, xs, cfg.state_dim, cfg.benettin_transient, cfg.benettin_measure)
    zero = torch.zeros(cfg.d_model, dtype=torch.float64, device=DEVICE)
    sp_phi = benettin_spectrum(lambda hh, x: cellf.phi(hh @ cellf.A.T.double()),
                               h0, [zero], cfg.state_dim, cfg.benettin_transient, cfg.benettin_measure)
    rr = model.cell.phi.get_r().detach().cpu().numpy()
    gam = model.cell.phi.get_gamma()
    s = dict(variant=variant, arm=arm, seed=seed, diverged=diverged,
             final_acc=hist[-1]["acc"] if hist else np.nan,
             lam_mean=float(np.mean([h["lam"] for h in hist])) if hist else np.nan,
             lam_final=hist[-1]["lam"] if hist else np.nan,
             lam_frac_in_corridor=float(np.mean([cfg.lam_low <= h["lam"] <= cfg.lam_high
                                                 for h in hist])) if hist else np.nan,
             lyap_share_mean=float(np.mean([h["lyap_share"] for h in hist])) if hist else np.nan,
             gnorm_max=float(np.max([h["gnorm"] for h in hist])) if hist else np.nan,
             clamp_rate=hist[-1]["clamp_rate"] if hist else np.nan,
             lam1_full=float(sp_full[0]), lam1_phi=float(sp_phi[0]),
             DKY_full=kaplan_yorke(sp_full), r_mean=float(rr.mean()),
             gamma=float(gam) if gam is not None else np.nan)
    return s, pd.DataFrame(hist), sp_full.cpu().numpy()

## 5. Stage B — variant screen

Each viable variant, corridor arm only. The question is narrow: **does λ̂ enter the corridor,
and does D_KY exceed 1?**

In [9]:
run_variants = VIABLE if VIABLE else list(CFG.variants)
if not VIABLE:
    print("[warn] Stage A found no viable variant; running all anyway for the record.")

S_rows, S_hist, S_spec = [], [], {}
for v in run_variants:
    for sd in range(CFG.screen_seeds):
        s, h, sp = train_one(v, "corridor", sd, CFG)
        S_rows.append(s); S_hist.append(h); S_spec[f"{v}_s{sd}"] = sp
        print(f"  {v:9s} seed={sd}  lam_mean={s['lam_mean']:+.4f}  "
              f"in_corridor={s['lam_frac_in_corridor']:.0%}  DKY={s['DKY_full']:.2f}  "
              f"clamp={s['clamp_rate']:.0%}  acc={s['final_acc']:.3f}")

SCR = pd.DataFrame(S_rows); SCRH = pd.concat(S_hist, ignore_index=True)
savetable(SCR, "phase0b_stageB_screen.csv"); savetable(SCRH, "phase0b_stageB_history.csv")
print()
print(SCR.groupby("variant")[["lam_mean","lam_frac_in_corridor","DKY_full",
                              "clamp_rate","lyap_share_mean","final_acc"]].mean().round(4).to_string())

  gain      seed=0  lam_mean=-0.1208  in_corridor=0%  DKY=0.00  clamp=68%  acc=0.065
  fold      seed=0  lam_mean=+0.7405  in_corridor=0%  DKY=21.42  clamp=3%  acc=0.060
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageB_screen.csv
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageB_history.csv

         lam_mean  lam_frac_in_corridor  DKY_full  clamp_rate  lyap_share_mean  final_acc
variant                                                                                  
fold       0.7405                   0.0   21.4239      0.0308           0.0125     0.0596
gain      -0.1208                   0.0    0.0000      0.6801           0.0002     0.0654


In [10]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for v in run_variants:
    d = SCRH[SCRH.variant == v].groupby("step").lam.mean()
    ax[0].plot(d.index, d.values, label=v)
ax[0].axhspan(CFG.lam_low, CFG.lam_high, color="green", alpha=.15, label="corridor")
ax[0].axhline(0, color="k", ls=":")
ax[0].set(xlabel="step", ylabel=r"$\hat\lambda$", title=r"$\hat\lambda$ by map variant")
ax[0].legend(); ax[0].grid(alpha=.3)

SCR.boxplot(column="DKY_full", by="variant", ax=ax[1]); ax[1].axhline(1.0, color="r", ls="--")
ax[1].set(title=r"$D_{KY}$ (full recurrence)", xlabel="variant", ylabel=r"$D_{KY}$")

SCR.boxplot(column="clamp_rate", by="variant", ax=ax[2])
ax[2].set(title="clamp activation rate", xlabel="variant", ylabel="rate")
fig.suptitle(""); fig.tight_layout(); savefig(fig, "phase0b_stageB_screen.png"); plt.close(fig)

score = SCR.groupby("variant").apply(
    lambda g: pd.Series(dict(in_corridor=g.lam_frac_in_corridor.mean(),
                             dky=g.DKY_full.mean())))
print(score.round(3).to_string())
WINNER = score.sort_values(["in_corridor","dky"], ascending=False).index[0]
print("\nStage B winner:", WINNER)

saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageB_screen.png
         in_corridor     dky
variant                     
fold             0.0  21.424
gain             0.0   0.000

Stage B winner: fold


/tmp/ipykernel_1584/3086514882.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  score = SCR.groupby("variant").apply(


## 6. Stage C — arm comparison on the winning variant

Only now is the H₅ question (corridor vs. zero-target vs. unregularized) meaningful, because
only now can the corridor actually be occupied.

In [11]:
C_rows, C_hist = [], []
for arm in CFG.arms:
    for sd in range(CFG.seeds):
        s, h, sp = train_one(WINNER, arm, sd, CFG)
        C_rows.append(s); C_hist.append(h)
        print(f"  {arm:9s} seed={sd}  lam_mean={s['lam_mean']:+.4f}  "
              f"in_corridor={s['lam_frac_in_corridor']:.0%}  DKY={s['DKY_full']:.2f}  "
              f"acc={s['final_acc']:.3f}")

SUM = pd.DataFrame(C_rows); HIST = pd.concat(C_hist, ignore_index=True)
savetable(SUM, "phase0b_stageC_arms.csv"); savetable(HIST, "phase0b_stageC_history.csv")
print()
print(SUM.groupby("arm")[["lam_mean","lam_frac_in_corridor","DKY_full",
                          "final_acc","lyap_share_mean"]].mean().round(4).to_string())

  corridor  seed=0  lam_mean=+0.7405  in_corridor=0%  DKY=21.42  acc=0.060
  corridor  seed=1  lam_mean=+0.5815  in_corridor=0%  DKY=21.36  acc=0.054
  floss     seed=0  lam_mean=+0.7400  in_corridor=0%  DKY=27.33  acc=0.062
  floss     seed=1  lam_mean=+0.5488  in_corridor=0%  DKY=24.97  acc=0.056
  free      seed=0  lam_mean=+0.6526  in_corridor=0%  DKY=20.67  acc=0.064
  free      seed=1  lam_mean=+0.5829  in_corridor=0%  DKY=17.78  acc=0.052
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageC_arms.csv
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageC_history.csv

          lam_mean  lam_frac_in_corridor  DKY_full  final_acc  lyap_share_mean
arm                                                                           
corridor    0.6610                   0.0   21.3914     0.0566           0.0096
floss       0.6444                   0.0   26.1537     0.0586           0.0147
free        0.6177                   0.0   19.2270     0.0581           0.

In [12]:
def paired(a, b, col="lam_mean"):
    x = SUM[SUM.arm==a].sort_values("seed")[col].values
    y = SUM[SUM.arm==b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(pair=f"{a} vs {b}", n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(pair=f"{a} vs {b}", n=len(x), p=float(p), median_diff=float(np.median(x-y)))

ST = pd.DataFrame([paired("corridor","free"), paired("corridor","floss"), paired("floss","free")])
print(ST.to_string(index=False)); savetable(ST, "phase0b_stageC_stats.csv")
pmin = 2/(2**CFG.seeds); print(f"\nmin achievable two-sided p at n={CFG.seeds}: {pmin:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for arm in CFG.arms:
    ax[0].hist(HIST[HIST.arm==arm].lam, bins=40, alpha=.5, label=arm, density=True)
ax[0].axvspan(CFG.lam_low, CFG.lam_high, color="green", alpha=.15, label="corridor")
ax[0].axvline(0, color="k", ls=":")
ax[0].set(xlabel=r"$\hat\lambda$", ylabel="density", title=f"{WINNER}: $\\hat\\lambda$ distribution")
ax[0].legend(); ax[0].grid(alpha=.3)
for arm in CFG.arms:
    d = HIST[HIST.arm==arm].groupby("step").lam.agg(["mean","std"])
    ax[1].plot(d.index, d["mean"], label=arm)
    ax[1].fill_between(d.index, d["mean"]-d["std"], d["mean"]+d["std"], alpha=.2)
ax[1].axhspan(CFG.lam_low, CFG.lam_high, color="green", alpha=.15); ax[1].axhline(0, color="k", ls=":")
ax[1].set(xlabel="step", ylabel=r"$\hat\lambda$", title="trajectory (mean ± sd)")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0b_stageC_arms.png"); plt.close(fig)

             pair  n   p  median_diff
 corridor vs free  2 NaN     0.043283
corridor vs floss  2 NaN     0.016593
    floss vs free  2 NaN     0.026690
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageC_stats.csv

min achievable two-sided p at n=2: 0.5000
saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_stageC_arms.png


## 7. Verdict

| Stage A | Stage B | Reading |
|---|---|---|
| `original` ceiling ≤ 0 | — | Confirms the Phase 0 result analytically. §4.3 Option A must be replaced, not retuned. |
| ≥1 variant usable, corridor occupied, D_KY > 1 | | The mechanism is viable. Proceed, with §4.3/§4.4 corrected. |
| variant usable but corridor never occupied | | The map *can* reach criticality and training declines to. That is a finding about the LM objective, and the expressivity argument in §3.2 needs defending. |
| corridor occupied but D_KY ≈ 1 | | Positive λ̂ without a strange attractor — check whether the estimator is being gamed by a single expanding direction. |
| `fold` reduces clamp rate but destabilizes training | | The state-domain fix works; the trade-off moves to trainability. Revisit `bptt_k`. |

Whatever the outcome, three corrections to the proposal are already established and should go
into the manuscript regardless: Option A's ceiling, the single-step estimator bias, and the
φ-vs-full-recurrence mismatch in §4.4.

In [13]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    stageA=PRE.to_dict("records"),
    viable_variants=VIABLE,
    stageB=SCR.groupby("variant")[["lam_mean","lam_frac_in_corridor","DKY_full","clamp_rate"]]
             .mean().round(4).to_dict("index"),
    winner=WINNER,
    stageC=SUM.groupby("arm")[["lam_mean","lam_frac_in_corridor","DKY_full","final_acc"]]
             .mean().round(4).to_dict("index"),
    stageC_stats=ST.to_dict("records"),
    config=dict(reg_target=CFG.reg_target, estimator=CFG.estimator,
                lyap_alpha=CFG.lyap_alpha, lyap_window=CFG.lyap_window),
)
savejson(verdict, "phase0b_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase0b/phase0b_verdict.json
{
  "smoke_test": true,
  "stageA": [
    {
      "variant": "original",
      "ceiling_logsigma": -0.029173322366263035,
      "typical_median": -1.073202675596908,
      "typical_p90": -1.0600006994514815,
      "reachable": false,
      "usable": false
    },
    {
      "variant": "gain",
      "ceiling_logsigma": 1.357121014332844,
      "typical_median": 1.283614491312944,
      "typical_p90": 1.2935186337860836,
      "reachable": true,
      "usable": true
    },
    {
      "variant": "fold",
      "ceiling_logsigma": 1.3571210387563946,
      "typical_median": 1.3114519285896256,
      "typical_p90": 1.329278406877087,
      "reachable": true,
      "usable": true
    }
  ],
  "viable_variants": [
    "gain",
    "fold"
  ],
  "stageB": {
    "fold": {
      "lam_mean": 0.7405,
      "lam_frac_in_corridor": 0.0,
      "DKY_full": 21.4239,
      "clamp_rate": 0.0308
    },
    "gain": {
      "lam_mean"